In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
TRADING_DAYS = 252
FORECAST_HORIZON = 20

prices = pd.read_csv(
    "../data/raw/stock_prices_20260102.csv",
    index_col=0,
    parse_dates=True
)

returns = prices.pct_change().dropna()


In [3]:
def create_lag_features(series, lags=5):
    df = pd.concat(
        [series.shift(i) for i in range(1, lags + 1)],
        axis=1
    )
    df.columns = [f"lag_{i}" for i in range(1, lags + 1)]
    return df.dropna()


In [4]:
forecasted_returns = {}

for ticker in returns.columns:
    y = returns[ticker]
    X = create_lag_features(y)

    y = y.loc[X.index]

    split = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # next-period forecast
    last_features = X.iloc[-1].values.reshape(1, -1)
    next_return = model.predict(last_features)[0]

    forecasted_returns[ticker] = {
        "predicted_daily_return": next_return,
        "rmse": rmse
    }


c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py

In [5]:
forecast_df = pd.DataFrame(forecasted_returns).T
forecast_df["annualized_return"] = (
    forecast_df["predicted_daily_return"] * TRADING_DAYS
)

forecast_df


,predicted_daily_return,rmse,annualized_return
AAPL,0.000318,0.013124,0.080108
AMZN,0.001603,0.018468,0.403951
GOOGL,0.001087,0.017882,0.273801
META,0.003206,0.020039,0.807787
MSFT,0.001567,0.010652,0.394857
NVDA,0.003709,0.020057,0.934586
SPY,0.000587,0.006635,0.147894
TSLA,0.002196,0.032130,0.553461


In [6]:
comparison = pd.DataFrame({
    "historical_return": returns.mean() * TRADING_DAYS,
    "forecasted_return": forecast_df["annualized_return"]
})

comparison


,historical_return,forecasted_return
AAPL,0.298152,0.080108
AMZN,0.382957,0.403951
GOOGL,0.470087,0.273801
META,0.632396,0.807787
MSFT,0.270496,0.394857
NVDA,0.987150,0.934586
SPY,0.220530,0.147894
TSLA,0.656810,0.553461


The forecasted expected returns can replace historical averages
in the portfolio optimization process.

This allows comparison between:
- Backward-looking optimization (historical mean)
- Forward-looking optimization (forecast-based)


In [7]:
ROLLING_WINDOW = 60  

forecast_daily_returns = returns.rolling(
    window=ROLLING_WINDOW
).mean().iloc[-1]


In [8]:
forecast_annual_returns = forecast_daily_returns * TRADING_DAYS

forecast_df = pd.DataFrame({
    "annualized_return": forecast_annual_returns
})


In [10]:
forecast_df.to_csv(
    "../data/processed/forecasted_returns.csv"
)